# Gymnasium Environments with PPO

## Setup
We will use gymnasium and torch for this implementation.

In [1]:
ENV_ID = "Acrobot-v1"

Install packages:

In [2]:
%pip install gymnasium stable-baselines3 wandb tsilva-notebook-utils==0.0.121 --quiet

Note: you may need to restart the kernel to use updated packages.


Load secrets:

In [3]:
from tsilva_notebook_utils.colab import load_secrets_into_env

_ = load_secrets_into_env([
    'WANDB_API_KEY'
])

Retrieve enviroment variables:

In [4]:
import torch
import numpy as np
from tsilva_notebook_utils.torch import get_default_device

DEVICE = get_default_device()
DEVICE

device(type='mps')

Define training configuration:

In [ ]:
import torch.nn as nn
from tsilva_notebook_utils.gymnasium import build_env as _build_env, set_random_seed
from dataclasses import dataclass
from typing import Union, Tuple

@dataclass
class PPOConfig:
    # Environment
    env_id: str = "CartPole-v1"
    seed: int = 42
    
    # Training
    max_epochs: int = -1
    gamma: float = 0.99
    lam: float = 0.95
    clip_epsilon: float = 0.2
    minibatch_size: int = 64
    train_rollout_steps: int = 2048
    
    # Evaluation
    eval_interval: int = 10
    eval_episodes: int = 32
    reward_threshold: float = 200
    
    # Networks
    policy_lr: float = 3e-4
    value_lr: float = 1e-3
    hidden_dim: Union[int, Tuple[int, ...]] = 64
    entropy_coef: float = 0.01
    
    # Other
    normalize: bool = False
    mean_reward_window: int = 100
    rollout_interval: int = 10
    n_envs: Union[str, int] = "auto"
    async_rollouts: bool = True
    
    @classmethod
    def for_env(cls, env_id: str) -> 'PPOConfig':
        """Factory method for environment-specific configs"""
        base = cls(env_id=env_id)
        
        env_overrides = {
            "CartPole-v1": dict(
                train_rollout_steps=512,
                minibatch_size=256,
                rollout_interval=1,
                eval_interval=20,
                eval_episodes=5,
                reward_threshold=475,
                policy_lr=1e-3,
                value_lr=1e-3,
                hidden_dim=32,
            ),
            # NOTE: converged in 39 epochs, 287.63 seconds (4.79 minutes)
            "Acrobot-v1": dict(
                gamma=0.99,
                lam=0.98,  # Increased from 0.95 for better advantage estimation
                clip_epsilon=0.2,  # Increased back to 0.2 for less conservative updates
                minibatch_size=128,  # Increased for more stable gradients
                train_rollout_steps=2048,  # Reduced from 4096 for faster iterations
                eval_interval=5,  # More frequent evaluation
                reward_threshold=-100,
                policy_lr=3e-4,  # Increased learning rates for faster learning
                value_lr=3e-4,
                hidden_dim=(128, 64),  # Slightly smaller network
                entropy_coef=0.01,  # Reduced entropy for more focused exploration
                rollout_interval=1
            ),
            "LunarLander-v3": dict(
                gamma=0.99,
                lam=0.95,
                clip_epsilon=0.2,
                minibatch_size=64,
                eval_interval=2,
                reward_threshold=200,
                policy_lr=1e-4,
                value_lr=5e-4,
                hidden_dim=32,
                entropy_coef=0.02
            ),
            "Pendulum-v1": dict(
                gamma=0.99,
                lam=0.95,
                clip_epsilon=0.2,
                minibatch_size=64,
                eval_interval=2,
                eval_episodes=5,
                reward_threshold=-200,
                policy_lr=3e-4,
                value_lr=1e-3,
                hidden_dim=(128, 64),
                entropy_coef=0.0
            ),
            "MountainCar-v0": dict(
                gamma=0.99,
                lam=0.97,
                clip_epsilon=0.15,
                minibatch_size=16,
                eval_interval=2,
                eval_episodes=10,
                reward_threshold=-110,
                policy_lr=1e-4,
                value_lr=5e-4,
                hidden_dim=(128, 64),
                entropy_coef=0.05
            ),
        }
        
        if env_id in env_overrides:
            for key, value in env_overrides[env_id].items():
                setattr(base, key, value)
        
        return base

CONFIG = PPOConfig.for_env(ENV_ID)
CONFIG

PPOConfig(env_id='Acrobot-v1', seed=42, max_epochs=-1, gamma=0.99, lam=0.98, clip_epsilon=0.2, minibatch_size=128, train_rollout_steps=2048, eval_interval=5, eval_episodes=32, reward_threshold=-100, policy_lr=0.0003, value_lr=0.0003, hidden_dim=(128, 64), entropy_coef=0.01, normalize=False, mean_reward_window=100, rollout_interval=1, n_envs='auto', async_rollouts=True)

Build environment:

In [6]:
from tsilva_notebook_utils.gymnasium import log_env_info

# Set random seed for reproducibility
set_random_seed(CONFIG.seed)

# Wrap build env with config parameters
build_env = lambda seed, n_envs=None: _build_env(
    CONFIG.env_id, 
    norm_obs=CONFIG.normalize, 
    n_envs=n_envs if n_envs is not None else CONFIG.n_envs, 
    seed=seed
)

# Test building env
env = build_env(CONFIG.seed)
log_env_info(env)

Environment Info (SubprocVecEnv with 8 envs)
  Env ID: Acrobot-v1
  Observation space: Box(low=[-1, -1, -1, -1, -12.6, -28.3], high=[1, 1, 1, 1, 12.6, 28.3], shape=(6,), dtype=float32)
  Action space: Discrete(3)
  Max episode steps: 500


## Build Agent

Define models:

In [7]:
class MLPNet(nn.Module):
    """Reusable MLP with configurable hidden dimensions"""
    
    def __init__(self, input_dim, output_dim, hidden_dim=64, activation=nn.ReLU):
        super().__init__()
        
        if isinstance(hidden_dim, (int, float)):
            hidden_dims = [int(hidden_dim)]
        else:
            hidden_dims = [int(dim) for dim in hidden_dim]
        
        layers = []
        current_dim = input_dim
        
        for hidden_size in hidden_dims:
            layers.extend([
                nn.Linear(current_dim, hidden_size),
                activation()
            ])
            current_dim = hidden_size
        
        layers.append(nn.Linear(current_dim, output_dim))
        self.net = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.net(x)

class PolicyNet(MLPNet):
    def __init__(self, obs_dim, act_dim, hidden_dim=64):
        super().__init__(obs_dim, act_dim, hidden_dim)

class ValueNet(MLPNet):
    def __init__(self, obs_dim, hidden_dim=64):
        super().__init__(obs_dim, 1, hidden_dim)

Define PPO loss function:

In [8]:
from torch.distributions import Categorical

class PPOLoss:
    def __init__(self, clip_epsilon, entropy_coef):
        self.clip_epsilon = clip_epsilon
        self.entropy_coef = entropy_coef
    
    def compute(self, states, actions, old_logps, advantages, returns, policy_model, value_model):
        # Policy loss
        logits = policy_model(states)
        dist = Categorical(logits=logits)
        new_logps = dist.log_prob(actions)
        
        ratio = torch.exp(new_logps - old_logps)
        surr1 = ratio * advantages
        surr2 = torch.clamp(ratio, 1.0 - self.clip_epsilon, 1.0 + self.clip_epsilon) * advantages
        entropy = dist.entropy().mean()
        
        policy_loss = -torch.min(surr1, surr2).mean() - self.entropy_coef * entropy
        
        # Value loss
        value_pred = value_model(states).squeeze()
        value_loss = 0.5 * ((returns - value_pred) ** 2).mean()
        
        # Metrics
        clip_fraction = ((ratio < 1.0 - self.clip_epsilon) | (ratio > 1.0 + self.clip_epsilon)).float().mean()
        kl_div = (old_logps - new_logps).mean()
        approx_kl = ((ratio - 1) - torch.log(ratio)).mean()
        explained_var = 1 - torch.var(returns - value_pred) / torch.var(returns)
        
        return {
            'policy_loss': policy_loss,
            'value_loss': value_loss,
            'entropy': entropy,
            'clip_fraction': clip_fraction,
            'kl_div': kl_div,
            'approx_kl': approx_kl,
            'explained_var': explained_var
        }

Create generic metric tracker:

In [9]:
class MetricTracker:
    """Unified metric collection and logging system"""
    
    def __init__(self, logger=None):
        self.logger = logger
        self.reset()
    
    def reset(self):
        """Reset epoch-level metrics"""
        self.step_metrics = []
    
    def add_step_metrics(self, metrics_dict):
        """Add metrics from a single training step"""
        self.step_metrics.append({k: v.detach() if hasattr(v, 'detach') else v 
                                 for k, v in metrics_dict.items()})
    
    def compute_epoch_means(self):
        """Compute mean of all step metrics for the epoch"""
        if not self.step_metrics:
            return {}
        
        epoch_metrics = {}
        for key in self.step_metrics[0].keys():
            values = [m[key] for m in self.step_metrics]
            epoch_metrics[key] = torch.stack(values).mean() if hasattr(values[0], 'dim') else np.mean(values)
        
        return epoch_metrics
    
    def log_metrics(self, metrics_dict, prefix="", prog_bar=False):
        """Log metrics with optional prefix"""
        if not self.logger:
            return
            
        formatted_metrics = {}
        for key, value in metrics_dict.items():
            if value is not None:
                full_key = f"{prefix}/{key}" if prefix else key
                formatted_metrics[full_key] = value
        
        if formatted_metrics:
            self.logger.log_dict(formatted_metrics, prog_bar=prog_bar)
    
    def log_single(self, key, value, prog_bar=False):
        """Log a single metric"""
        if self.logger and value is not None:
            self.logger.log(key, value, prog_bar=prog_bar)

Define rollout collectors:

In [10]:
import time
import multiprocessing
import threading
import queue
import copy
from collections import deque
from tsilva_notebook_utils.gymnasium import RolloutDataset, collect_rollouts, group_trajectories_by_episode

class BaseRolloutCollector:
    """Base class for rollout collectors"""
    def __init__(self, build_env_fn, config, obs_dim, act_dim):
        self.build_env_fn = build_env_fn
        self.config = config
        self.obs_dim = obs_dim
        self.act_dim = act_dim
        self.last_obs = None
        
    def start(self):
        """Start the collector"""
        pass
        
    def stop(self):
        """Stop the collector"""
        pass
        
    def update_models(self, policy_state_dict, value_state_dict):
        """Update model weights"""
        pass
        
    def get_rollout(self, timeout=1.0):
        """Get next rollout data"""
        raise NotImplementedError
        
    def initialize_with_models(self, policy_model, value_model):
        """Initialize collector with model references"""
        pass

In [11]:
class SyncRolloutCollector(BaseRolloutCollector):
    """Synchronous rollout collector - collects data on demand"""
    def __init__(self, build_env_fn, config, obs_dim, act_dim):
        super().__init__(build_env_fn, config, obs_dim, act_dim)
        self.env = build_env_fn(config.seed)
        self.policy_model = None
        self.value_model = None
        self._ready_for_initial = False
        
    def initialize_with_models(self, policy_model, value_model):
        """Set model references for sync collector"""
        self.policy_model = policy_model
        self.value_model = value_model
        self._ready_for_initial = True
        
    def get_rollout(self, timeout=1.0):
        """Collect rollout synchronously using current models"""
        if self.policy_model is None or self.value_model is None:
            return None
            
        trajectories, extras = collect_rollouts(
            self.env,
            self.policy_model,
            self.value_model,
            n_steps=self.config.train_rollout_steps,
            last_obs=self.last_obs
        )
        
        self.last_obs = extras['last_obs']
        return trajectories
        
    def is_ready_for_initial_rollout(self):
        """Check if ready for initial rollout collection"""
        return self._ready_for_initial

In [12]:
class AsyncRolloutCollector(BaseRolloutCollector):
    """Background thread that continuously collects rollouts using latest model weights"""
    def __init__(self, build_env_fn, config, obs_dim, act_dim):
        super().__init__(build_env_fn, config, obs_dim, act_dim)
        
        # Thread-safe queue for rollout data
        self.rollout_queue = queue.Queue(maxsize=3)  # Buffer 3 rollouts max
        
        # Shared model weights (CPU copies for thread safety)
        self.policy_state_dict = None
        self.value_state_dict = None
        self.model_lock = threading.Lock()
        
        # Control flags
        self.running = False
        self.thread = None
        
        # Create environment and models for rollout collection
        self.env = None
        self.policy_model = None
        self.value_model = None
        
    def initialize_with_models(self, policy_model, value_model):
        """Initialize with model state dicts for async collector"""
        self.update_models(policy_model.state_dict(), value_model.state_dict())
        
    def start(self):
        """Start the background rollout collection thread"""
        if self.running:
            return
            
        self.running = True
        self.thread = threading.Thread(target=self._collect_loop, daemon=True)
        self.thread.start()
        
    def stop(self):
        """Stop the background rollout collection"""
        self.running = False
        if self.thread:
            self.thread.join(timeout=5.0)
            
    def update_models(self, policy_state_dict, value_state_dict):
        """Update model weights from main training thread"""
        with self.model_lock:
            self.policy_state_dict = copy.deepcopy(policy_state_dict)
            self.value_state_dict = copy.deepcopy(value_state_dict)
            
    def get_rollout(self, timeout=1.0):
        """Get next rollout data (non-blocking with timeout)"""
        try:
            return self.rollout_queue.get(timeout=timeout)
        except queue.Empty:
            return None
            
    def is_ready_for_initial_rollout(self):
        """Check if ready for initial rollout collection"""
        return self.policy_state_dict is not None and self.value_state_dict is not None

    def _init_models(self):
        """Initialize models in the worker thread"""
        if self.env is None:
            self.env = self.build_env_fn(self.config.seed + 1000)  # Different seed for rollout env
        
        if self.policy_model is None:
            self.policy_model = PolicyNet(self.obs_dim, self.act_dim, self.config.hidden_dim)
            self.policy_model.eval()  # Always in eval mode for rollouts
            
        if self.value_model is None:
            self.value_model = ValueNet(self.obs_dim, self.config.hidden_dim)
            self.value_model.eval()
            
    def _update_model_weights(self):
        """Update local model weights from shared state dicts"""
        with self.model_lock:
            if self.policy_state_dict is not None:
                self.policy_model.load_state_dict(self.policy_state_dict)
            if self.value_state_dict is not None:
                self.value_model.load_state_dict(self.value_state_dict)
                
    def _collect_loop(self):
        """Main loop running in background thread"""
        self._init_models()
        
        while self.running:
            try:
                # Update to latest model weights
                self._update_model_weights()
                
                # Collect rollout
                trajectories, extras = collect_rollouts(
                    self.env,
                    self.policy_model,
                    self.value_model,
                    n_steps=self.config.train_rollout_steps,
                    last_obs=self.last_obs
                )
                
                self.last_obs = extras['last_obs']
                
                # Put rollout in queue (non-blocking, drop if full)
                try:
                    self.rollout_queue.put(trajectories, block=False)
                except queue.Full:
                    # Queue is full, drop oldest and add new
                    try:
                        self.rollout_queue.get_nowait()
                        self.rollout_queue.put(trajectories, block=False)
                    except queue.Empty:
                        pass
                        
            except Exception as e:
                print(f"Error in rollout collection: {e}")
                time.sleep(0.1)  # Brief pause on error

Define agent class:

In [ ]:
import pytorch_lightning as pl
from torch.utils.data import DataLoader

# ---------------------------------------------------------------------
class Agent(pl.LightningModule):
    """Base agent class with common RL functionality"""
    
    def __init__(self, obs_dim, act_dim, config, build_env_fn):
        super().__init__()
        
        # Store core attributes
        self.obs_dim = obs_dim
        self.act_dim = act_dim
        self.config = config
        self.build_env_fn = build_env_fn
        
        # Common RL components
        self.env = build_env_fn(config.seed)
        self.metrics = MetricTracker(self)
        self.rollout_ds = RolloutDataset()
        self.episode_reward_deque = deque(maxlen=config.mean_reward_window)
        
        # Rollout collection
        rollout_collector_cls = AsyncRolloutCollector if config.async_rollouts else SyncRolloutCollector
        self.rollout_collector = rollout_collector_cls(build_env_fn, config, obs_dim, act_dim)
        
        # Training state
        self.automatic_optimization = False
        self.training_start_time = None
        
    def create_models(self):
        """Override in subclass to create algorithm-specific models"""
        raise NotImplementedError("Subclass must implement create_models()")
        
    def compute_loss(self, batch):
        """Override in subclass to compute algorithm-specific loss"""
        raise NotImplementedError("Subclass must implement compute_loss()")
        
    def optimize_models(self, loss_results):
        """Override in subclass to implement algorithm-specific optimization"""
        raise NotImplementedError("Subclass must implement optimize_models()")
        
    def get_models_for_rollout(self):
        """Override in subclass to return models needed for rollout collection"""
        raise NotImplementedError("Subclass must implement get_models_for_rollout()")

    def setup(self, stage: str):
        if stage == "fit":
            policy_model, value_model = self.get_models_for_rollout()
            self.rollout_collector.initialize_with_models(policy_model, value_model)
            self.rollout_collector.start()
            
            print("Waiting for initial rollout...")
            while True:
                if self.rollout_collector.is_ready_for_initial_rollout():
                    trajectories = self.rollout_collector.get_rollout(timeout=2.0)
                    if trajectories is not None:
                        self._update_rollout_data(trajectories)
                        break
                print("Still waiting for rollout...")

    def train_dataloader(self):
        return DataLoader(
            self.rollout_ds,
            batch_size=self.config.minibatch_size,
            shuffle=True,
            pin_memory=True if self.device.type != 'mps' else False,
            num_workers=multiprocessing.cpu_count() // 2# if self.device.type != 'mps' else 0
        )

    def on_fit_start(self):
        self.training_start_time = time.time()
        mode = "async" if self.config.async_rollouts else "sync"
        print(f"Training started in {mode} mode at {time.strftime('%Y-%m-%d %H:%M:%S')}")
    
    def on_fit_end(self):
        self.rollout_collector.stop()
        if self.training_start_time:
            total_time = time.time() - self.training_start_time
            print(f"Training completed in {total_time:.2f} seconds ({total_time/60:.2f} minutes)")

    def on_train_epoch_start(self):
        self.metrics.reset()
        policy_model, value_model = self.get_models_for_rollout()
        self.rollout_collector.update_models(
            policy_model.state_dict(), value_model.state_dict()
        )
        
        # Collect new rollout if needed
        if (self.current_epoch + 1) % self.config.rollout_interval == 0:
            self._collect_and_update_rollout()

    def on_train_epoch_end(self):
        # Log epoch metrics
        epoch_metrics = self.metrics.compute_epoch_means()
        if epoch_metrics:
            self.metrics.log_metrics(epoch_metrics, prefix="epoch")
        
        # Evaluation
        if (self.current_epoch + 1) % self.config.eval_interval == 0:
            self._evaluate_and_check_stopping()

    def training_step(self, batch, batch_idx):
        states, actions, rewards, dones, old_logps, values, advantages, returns, frames = batch

        # Compute algorithm-specific losses and metrics
        loss_results = self.compute_loss(batch)

        # Track step metrics
        self.metrics.add_step_metrics(loss_results)

        # Optimize models
        self.optimize_models(loss_results)

        # Log training metrics
        self._log_training_metrics(loss_results, advantages, values, returns)

        return sum(loss for key, loss in loss_results.items() if 'loss' in key)

    def _collect_and_update_rollout(self):
        """Collect and update rollout data"""
        timeout = 2.0 if self.config.async_rollouts else 1.0
        trajectories = self.rollout_collector.get_rollout(timeout=timeout)
        
        if trajectories is not None:
            self._update_rollout_data(trajectories)
            self.metrics.log_single('rollout/queue_updated', 1.0)
        else:
            self.metrics.log_single('rollout/queue_miss', 1.0)
    
    def _update_rollout_data(self, trajectories):
        """Update rollout dataset and episode rewards"""
        self.rollout_ds.update(*trajectories)
        episodes = group_trajectories_by_episode(trajectories)
        episode_rewards = [sum(step[2] for step in episode) for episode in episodes]
        for r in episode_rewards:
            self.episode_reward_deque.append(float(r))

    def _log_training_metrics(self, loss_results, advantages, values, returns):
        """Log common training metrics"""
        mean_reward = np.mean(self.episode_reward_deque) if len(self.episode_reward_deque) > 0 else 0
        
        # Core metrics that most algorithms will have
        train_metrics = {
            'mean_reward': mean_reward,
        }
        
        # Add algorithm-specific loss metrics
        for key, value in loss_results.items():
            if 'loss' in key or key in ['entropy', 'kl_divergence', 'explained_variance']:
                train_metrics[key] = value
        
        additional_metrics = {
            'advantage_mean': advantages.mean(),
            'advantage_std': advantages.std(),
            'value_mean': values.mean(),
            'returns_mean': returns.mean(),
        }
        
        # Add any additional algorithm-specific metrics
        for key, value in loss_results.items():
            if key not in train_metrics and key not in additional_metrics:
                additional_metrics[key] = value
        
        self.metrics.log_metrics(train_metrics, prefix="train", prog_bar=True)
        self.metrics.log_metrics(additional_metrics, prefix="train", prog_bar=False)

    def _evaluate_and_check_stopping(self):
        """Evaluate model and check for early stopping"""
        eval_seed = np.random.randint(0, 1_000_000)
        eval_env = self.build_env_fn(eval_seed)
        
        policy_model, value_model = self.get_models_for_rollout()
        policy_model.eval()
        try:
            eval_mean_reward = self._run_evaluation(eval_env, policy_model, value_model)
            self.metrics.log_single('eval/mean_reward', eval_mean_reward, prog_bar=True)
            
            if eval_mean_reward >= self.config.reward_threshold:
                print(f"Early stopping at epoch {self.current_epoch} with eval mean reward {eval_mean_reward:.2f} >= threshold {self.config.reward_threshold}")
                self.trainer.should_stop = True
                
        finally:
            policy_model.train()
            eval_env.close()

    def _run_evaluation(self, env, policy_model, value_model):
        """Run evaluation and log rollout metrics"""
        start = time.time()
        trajectories, _ = collect_rollouts(
            env, policy_model, value_model,
            n_episodes=self.config.eval_episodes, deterministic=False
        )
        elapsed = time.time() - start

        episodes = group_trajectories_by_episode(trajectories)
        episode_rewards = [sum(step[2] for step in episode) for episode in episodes]
        mean_episode_reward = np.mean(episode_rewards)

        # Log rollout metrics
        rollout_metrics = {
            'mean_reward': mean_episode_reward,
            'num_episodes': len(episodes),
            'num_steps': len(trajectories[0]),
            'avg_steps_per_episode': len(trajectories[0]) / (len(episodes) + 1e-3),
            'time_elapsed': elapsed,
            'steps_per_second': len(trajectories[0]) / (elapsed + 1e-3)
        }
        
        self.metrics.log_metrics(rollout_metrics, prefix="rollout")
        return mean_episode_reward


In [14]:
class PPOAgent(Agent):
    """PPO-specific agent implementation"""
    
    def __init__(self, obs_dim, act_dim, config, build_env_fn):
        super().__init__(obs_dim, act_dim, config, build_env_fn)
        self.save_hyperparameters(ignore=['build_env_fn'])
        
        # Create PPO-specific models and components
        self.create_models()
        self.ppo_loss = PPOLoss(config.clip_epsilon, config.entropy_coef)
        
    def create_models(self):
        """Create PPO-specific policy and value models"""
        self.policy_model = PolicyNet(self.obs_dim, self.act_dim, self.config.hidden_dim)
        self.value_model = ValueNet(self.obs_dim, self.config.hidden_dim)
        
    def get_models_for_rollout(self):
        """Return models needed for rollout collection"""
        return self.policy_model, self.value_model
        
    def compute_loss(self, batch):
        """Compute PPO-specific losses"""
        states, actions, rewards, dones, old_logps, values, advantages, returns, frames = batch
        
        return self.ppo_loss.compute(
            states, actions, old_logps, advantages, returns, 
            self.policy_model, self.value_model
        )
        
    def optimize_models(self, loss_results):
        """Optimize PPO policy and value models"""
        opt_policy, opt_value = self.optimizers()
        
        # Optimize policy
        opt_policy.zero_grad()
        self.manual_backward(loss_results['policy_loss'])
        opt_policy.step()

        # Optimize value function
        opt_value.zero_grad()
        self.manual_backward(loss_results['value_loss'])
        opt_value.step()

    def configure_optimizers(self):
        return [
            torch.optim.Adam(self.policy_model.parameters(), lr=self.config.policy_lr),
            torch.optim.Adam(self.value_model.parameters(), lr=self.config.value_lr)
        ]

    def forward(self, x):
        return self.policy_model(x)

## Train

In [15]:
import os
import sys, builtins, wandb
from typing import Any
from IPython import get_ipython
import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger
from wandb.errors import CommError
try:
    from wandb.sdk.mailbox.mailbox import MailboxClosedError
except ImportError:
    MailboxClosedError = RuntimeError


class WandbCleanup(pl.Callback):
    """Notebook-safe cleanup for W&B + PyTorch Lightning."""
    def __init__(self) -> None:
        super().__init__()
        os.environ["WANDB_SILENT"] = "True"    # before wandb.init / Lightning start
        self._done = False

    # internal helper ---------------------------------------------------
    def _close_wandb(self, trainer: pl.Trainer) -> None:
        if self._done:
            return
        self._done = True

        # 1) finish any active run(s)
        loggers = (
            trainer.loggers
            if hasattr(trainer, "loggers")
            else ([trainer.logger] if trainer.logger else [])
        )
        for lg in loggers:
            if isinstance(lg, WandbLogger):
                run = getattr(lg, "experiment", None)
                if run and not getattr(run, "_is_finished", False):
                    try:
                        run.finish(exit_code=-1)
                    # swallow every “backend already dead” variant
                    except (MailboxClosedError, CommError, BrokenPipeError, OSError):
                        pass      
                    except Exception as e:
                        print(f"[WandbCleanup] run.finish() failed: {e}")

        # 2) global teardown
        try:
            wandb.teardown()
        except Exception:
            pass

        # 3) restore exit/quit
        builtins.exit = sys.exit
        builtins.quit = sys.exit

        # 4) unregister IPython hooks
        try:
            ip = get_ipython()
            if ip and hasattr(ip, "events"):
                for evt in ("pre_run_cell", "post_run_cell"):
                    for cb in list(ip.events.callbacks.get(evt, [])):
                        if getattr(cb, "__module__", "").startswith("wandb"):
                            ip.events.unregister(evt, cb)
        except Exception:
            pass

    # Lightning-dispatched hooks ---------------------------------------
    def on_keyboard_interrupt(self, trainer: pl.Trainer, *_: Any) -> None:
        self._close_wandb(trainer)

    def on_exception(
        self,
        trainer: pl.Trainer,
        pl_module: pl.LightningModule,
        exception: BaseException,          # <— positional, fixes TypeError
    ) -> None:
        self._close_wandb(trainer)

    def on_fit_end(self, trainer: pl.Trainer, *_: Any) -> None:
        self._close_wandb(trainer)

    def on_validation_end(self, trainer: pl.Trainer, *_: Any) -> None:
        self._close_wandb(trainer)

    def on_test_end(self, trainer: pl.Trainer, *_: Any) -> None:
        self._close_wandb(trainer)

    def on_predict_end(self, trainer: pl.Trainer, *_: Any) -> None:
        self._close_wandb(trainer)

In [16]:

from pytorch_lightning.loggers import WandbLogger

# Create PPO agent
obs_dim = env.observation_space.shape[0]
act_dim = env.action_space.n if hasattr(env.action_space, 'n') else env.action_space.shape[0]
ppo_agent = PPOAgent(obs_dim, act_dim, CONFIG, build_env)

# Set up trainer with proper device configuration
wandb_logger = WandbLogger(project=ENV_ID)

# Print W&B run URL explicitly
print(f"🔗 W&B Run: {wandb_logger.experiment.url}")

trainer = pl.Trainer(
    logger=wandb_logger,
    log_every_n_steps=10,
    max_epochs=CONFIG.max_epochs,
    enable_progress_bar=True,
    enable_checkpointing=False,  # Disable checkpointing for speed
    accelerator="auto",
    callbacks=[WandbCleanup()]
)

# Fit the model
trainer.fit(ppo_agent)

wandb: Currently logged in as: tsilva to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


🔗 W&B Run: https://wandb.ai/tsilva/Acrobot-v1/runs/6cnhbn6g
Waiting for initial rollout...
Still waiting for rollout...



  | Name         | Type      | Params | Mode 
---------------------------------------------------
0 | policy_model | PolicyNet | 9.3 K  | train
1 | value_model  | ValueNet  | 9.2 K  | train
---------------------------------------------------
18.6 K    Trainable params
0         Non-trainable params
18.6 K    Total params
0.074     Total estimated model params size (MB)
14        Modules in train mode
0         Modules in eval mode
/Users/tsilva/repos/tsilva/aiml-notebooks/.venv/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Training started in async mode at 2025-07-15 17:02:21
Epoch 39: 100%|██████████| 128/128 [00:10<00:00, 12.07it/s, v_num=bn6g, train/mean_reward=-93.6, train/policy_loss=0.142, train/value_loss=45.30, train/entropy=0.462, eval/mean_reward=-103.]

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


epoch,▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▆▆▆▇▇▇██
epoch/approx_kl,▂▃▂▂▂▁▂▁▃▃▂▂▄▄▄▄▅▃▄█▅▄▆▅▅▄▇▇▃▅▃▂▃▃▃▁▂▂▁▂
epoch/clip_fraction,▂▃▁▂▁▁▁▁▃▃▂▂▄▄▄▃▄▃▃█▄▄▅▅▅▄▇▇▄▅▃▃▃▃▃▁▂▃▂▂
epoch/entropy,█████████▇██▇█▇▇▇▇▆▆▆▆▅▅▅▄▄▄▃▃▃▂▂▂▂▂▁▁▁▁
epoch/explained_var,▆▁▄▆▆▇▇▇▇▇██████████████████████████████
epoch/kl_div,▂▂▂▃▂▁▂▁▃▃▂▂▅▅▅▃▅▃▅█▅▄▅▆▆▄▆█▃▅▄▃▃▃▃▁▁▂▁▁
epoch/policy_loss,▄▄▅▄▆▅▄▄▄▄▅▃▄▄▃▃▂▃▃▁▃▄▃▃▃▄▃▅▆▅▆▆▇▇▇▇▇▇██
epoch/value_loss,█▂▁▂▁▂▁▁▂▂▂▂▂▂▂▂▂▂▂▂▂▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▂▂
eval/mean_reward,▁▄▆▇▇███
rollout/avg_steps_per_episode,█▅▃▂▂▁▁▁
rollout/mean_reward,▁▄▆▇▇███


Training completed in 287.63 seconds (4.79 minutes)


## Evaluate

In [17]:
import random
from tsilva_notebook_utils.gymnasium import render_episode_frames

n_episodes = 8
trajectories, _ = collect_rollouts(
    build_env(random.randint(0, 1_000_000), n_envs=n_episodes),
    ppo_agent.policy_model,
    n_episodes=n_episodes,
    deterministic=True,
    collect_frames=True
)
episodes = group_trajectories_by_episode(trajectories) # something is wrong in frame collection
mean_reward = np.mean([sum(step[2] for step in episode) for episode in episodes])
episode_frames = [[step[-1] for step in episode] for episode in episodes]
print(f"Mean reward: {mean_reward:.2f}")
render_episode_frames(episode_frames, out_dir="./tmp", grid=(2, 2), text_color=(0, 0, 0))

/Users/tsilva/repos/tsilva/aiml-notebooks/.venv/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
/Users/tsilva/repos/tsilva/aiml-notebooks/.venv/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
/Users/tsilva/repos/tsilva/aiml-notebooks/.venv/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_

Mean reward: -99.00
